In [ ]:
#IMPORT DATASET

# File location and type
file_location = "./flight_weather.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df)

In [ ]:
print(df.count(), len(df.columns))

In [ ]:
df.printSchema()

In [ ]:
# ARR_DEL15 = 1 if it's canceled.
from pyspark.sql.functions import when
df = df.withColumn("ARR_DEL15", when(df["CANCELLED"] == 1, 1).otherwise(df["ARR_DEL15"]))

In [ ]:
df = df.filter(df["DIVERTED"] == 0)

In [ ]:
from pyspark.sql.types import IntegerType

df = df \
    .withColumns(
        {
            "RelativeHumidityOrigin": df["RelativeHumidityOrigin"].cast(IntegerType()),
            "AltimeterOrigin": df["AltimeterOrigin"].cast(IntegerType()),
            "DryBulbCelsiusOrigin": df["DryBulbCelsiusOrigin"].cast(IntegerType()),
            "WindSpeedOrigin": df["WindSpeedOrigin"].cast(IntegerType()),
            "VisibilityOrigin": df["VisibilityOrigin"].cast(IntegerType()),
            "DewPointCelsiusOrigin": df["DewPointCelsiusOrigin"].cast(IntegerType()),
            "RelativeHumidityDest": df["RelativeHumidityDest"].cast(IntegerType()),
            "AltimeterDest": df["AltimeterDest"].cast(IntegerType()),
            "DryBulbCelsiusDest": df["DryBulbCelsiusDest"].cast(IntegerType()),
            "WindSpeedDest": df["WindSpeedDest"].cast(IntegerType()),
            "VisibilityDest": df["VisibilityDest"].cast(IntegerType()),
            "DewPointCelsiusDest": df["DewPointCelsiusDest"].cast(IntegerType()),
            "ARR_DEL15": df["ARR_DEL15"].cast(IntegerType())
        }
    )


In [ ]:
df = df.select(
  "ARR_DEL15",
  "MONTH",
  "DAY_OF_WEEK",
  "UNIQUE_CARRIER",
  "ORIGIN",
  "DEST",
  "CRS_DEP_TIME",
  "CRS_ARR_TIME",
  "RelativeHumidityOrigin",
  "AltimeterOrigin",
  "DryBulbCelsiusOrigin",
  "WindSpeedOrigin",
  "VisibilityOrigin",
  "DewPointCelsiusOrigin",
  "RelativeHumidityDest",
  "AltimeterDest",
  "DryBulbCelsiusDest",
  "WindSpeedDest",
  "VisibilityDest",
  "DewPointCelsiusDest")

In [ ]:
df = df.dropna()

In [ ]:
print(df.count(), len(df.columns))

In [ ]:
summ = df.select("ARR_DEL15", "AltimeterDest", "WindSpeedDest", "AltimeterOrigin", "VisibilityDest", "WindSpeedOrigin", "VisibilityOrigin", "DryBulbCelsiusDest", "DewPointCelsiusDest", "DryBulbCelsiusOrigin", "RelativeHumidityDest", "DewPointCelsiusOrigin", "RelativeHumidityOrigin").summary(
"mean",
"stddev",
"min",
"1%",
"5%",
"50%",
"95%",
"99%",
"max",
)

display(summ)

In [ ]:
# Split data into train data and test data
(traindf, testdf) = df.randomSplit([0.8, 0.2])

In [ ]:
from pyspark.ml.feature import StringIndexer
uniqueCarrierIndexer = StringIndexer(inputCol="UNIQUE_CARRIER", outputCol="Indexed_UNIQUE_CARRIER").fit(df)
originIndexer = StringIndexer(inputCol="ORIGIN", outputCol="Indexed_ORIGIN").fit(df)
destIndexer = StringIndexer(inputCol="DEST", outputCol="Indexed_DEST").fit(df)
arrDel15Indexer = StringIndexer(inputCol="ARR_DEL15", outputCol="Indexed_ARR_DEL15").fit(df)

In [ ]:
# Assemble feature columns
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
  inputCols = [
    "MONTH",
    "DAY_OF_WEEK",
    "Indexed_UNIQUE_CARRIER",
    "Indexed_ORIGIN",
    "Indexed_DEST",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "RelativeHumidityOrigin",
    "AltimeterOrigin",
    "DryBulbCelsiusOrigin",
    "WindSpeedOrigin",
    "VisibilityOrigin",
    "DewPointCelsiusOrigin",
    "RelativeHumidityDest",
    "AltimeterDest",
    "DryBulbCelsiusDest",
    "WindSpeedDest",
    "VisibilityDest",
    "DewPointCelsiusDest"],
  outputCol = "features")

In [ ]:
# Generate classifier
from pyspark.ml.classification import DecisionTreeClassifier
classifier = DecisionTreeClassifier(featuresCol="features", labelCol="ARR_DEL15", maxDepth=15, maxBins=500)

In [ ]:
# Create pipeline and Train
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=[uniqueCarrierIndexer, originIndexer, destIndexer, arrDel15Indexer, assembler, classifier])
model = pipeline.fit(traindf)

In [ ]:
# Predict with eveluation data
pred = model.transform(testdf)

In [ ]:
# Evaluate results
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="ARR_DEL15", predictionCol="prediction")
accuracy = evaluator.evaluate(pred)
print("Accuracy = %g" % accuracy)